In [1]:
"""
Depletion width vs. bias, from Lumerical CHARGE monitor export(s).

Uses h5py to transparently handle v7.3/HDF5 .mat files from a native 
CHARGE "range" sweep, whether Lumerical stacked it as one struct 
(V_anode vector, n/p 2D [nodes x num_bias]) or exported it as a MATLAB 
struct array (one sub-struct per bias point).

Depletion edge criterion: majority-carrier concentration recovers to FRAC
(default 50%) of the local bulk doping level, read a fixed OFFSET away from
the metallurgical junction (n=p crossing).
"""
import glob
import h5py
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

In [2]:
X_WINDOW = 0.5    # um, half-width of the region around x=0 containing the junction
OFFSET   = 0.20   # um, distance from the junction used to read the "bulk" doping level
FRAC     = 0.50   # depletion edge = where carrier conc. recovers to FRAC * bulk level
Y_EVAL   = None   # um; height at height at which to evaluate the width. None -> mid-height

In [3]:
class SimpleNamespace:
    """Simple container to allow dot-notation access to dataset fields."""
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)


def load_charge_mat(matfile):
    """Loads Lumerical CHARGE data from a MATLAB v7.3 (.mat) file using h5py.
    Returns either a list of point objects (for struct arrays) or a namespace 
    containing the file's variables/fields.
    """
    with h5py.File(matfile, 'r') as f:
        keys = [k for k in f.keys() if not k.startswith('#')]
        
        base = f
        if len(keys) == 1 and isinstance(f[keys[0]], h5py.Group) and 'x' not in f:
            base = f[keys[0]]
            base_keys = [k for k in base.keys() if not k.startswith('#')]
        else:
            base_keys = keys

        # Check for a MATLAB struct array (stored as a dataset of object references)
        for k in base_keys:
            item = base[k]
            if isinstance(item, h5py.Dataset) and item.dtype == object:
                struct_list = []
                refs = item[()]
                for ref_row in np.atleast_1d(refs):
                    for ref in np.atleast_1d(ref_row):
                        if ref:
                            grp = f[ref]
                            pt_dict = {}
                            for field in grp.keys():
                                if not field.startswith('#'):
                                    sub_item = grp[field]
                                    val = sub_item[()] if isinstance(sub_item, h5py.Dataset) else sub_item
                                    pt_dict[field] = np.squeeze(val) if isinstance(val, np.ndarray) else val
                            struct_list.append(SimpleNamespace(**pt_dict))
                return struct_list

        # Single-struct or top-level layout
        data = {}
        for k in base_keys:
            item = base[k]
            if isinstance(item, h5py.Dataset):
                val = item[()]
                data[k] = np.squeeze(val) if isinstance(val, np.ndarray) else val
            elif isinstance(item, h5py.Group):
                if 'value' in item:
                    val = item['value'][()]
                    data[k] = np.squeeze(val) if isinstance(val, np.ndarray) else val
                else:
                    sub_dict = {}
                    for subk in item.keys():
                        if not subk.startswith('#'):
                            sub_item = item[subk]
                            if isinstance(sub_item, h5py.Dataset):
                                sub_val = sub_item[()]
                                sub_dict[subk] = np.squeeze(sub_val) if isinstance(sub_val, np.ndarray) else sub_val
                    data[k] = SimpleNamespace(**sub_dict)

        return SimpleNamespace(**data)


def width_at_y(interp_n, interp_p, yv, xq):
    nq = np.ma.filled(interp_n(xq, np.full_like(xq, yv)), np.nan)
    pq = np.ma.filled(interp_p(xq, np.full_like(xq, yv)), np.nan)
    if np.all(np.isnan(nq)):
        return np.nan
    j = np.nanargmin(np.abs(nq - pq))
    xj = xq[j]
    N_A = pq[np.nanargmin(np.abs(xq - (xj - OFFSET)))]
    N_D = nq[np.nanargmin(np.abs(xq - (xj + OFFSET)))]
    xp_edge = xj
    for k in range(j, -1, -1):
        if pq[k] >= FRAC * N_A:
            xp_edge = xq[k]; break
    xn_edge = xj
    for k in range(j, len(xq)):
        if nq[k] >= FRAC * N_D:
            xn_edge = xq[k]; break
    return (xn_edge - xp_edge) * 1000  # nm


def width_from_mesh(x_um, y_um, elements, n, p):
    x_um = np.ravel(x_um)
    y_um = np.ravel(y_um)
    
    elements = np.atleast_2d(elements)
    if elements.shape[0] == 3 and elements.shape[1] != 3:
        elements = elements.T

    tri = mtri.Triangulation(x_um, y_um, elements.astype(int) - 1)
    
    n_flat = np.ravel(n)
    p_flat = np.ravel(p)
    if len(n_flat) != len(x_um):
        raise ValueError(f"Concentration array n has length {len(n_flat)}, but mesh has {len(x_um)} nodes.")
    if len(p_flat) != len(x_um):
        raise ValueError(f"Concentration array p has length {len(p_flat)}, but mesh has {len(x_um)} nodes.")

    interp_n = mtri.LinearTriInterpolator(tri, n_flat)
    interp_p = mtri.LinearTriInterpolator(tri, p_flat)
    xq = np.linspace(-X_WINDOW, X_WINDOW, 2000)
    yv = Y_EVAL if Y_EVAL is not None else (y_um.min() + y_um.max()) / 2
    return width_at_y(interp_n, interp_p, yv, xq)


def process_file(matfile):
    """Returns (V_bias array, width array in nm) using direct h5py loading."""
    c = load_charge_mat(matfile)

    # --- struct-array layout: one sub-struct per bias point ---
    if isinstance(c, list):
        V_bias, widths = [], []
        for pt in c:
            V_bias.append(float(np.ravel(pt.V_anode)[0]) - float(np.ravel(pt.V_cathode)[0]))
            widths.append(width_from_mesh(pt.x * 1e6, pt.y * 1e6, pt.elements, pt.n, pt.p))
        order = np.argsort(V_bias)
        return np.array(V_bias)[order], np.array(widths)[order]

    # --- single-struct layout: shared mesh, n/p possibly 2D ---
    x_um, y_um = np.ravel(c.x) * 1e6, np.ravel(c.y) * 1e6
    elements = c.elements
    num_nodes = len(x_um)

    V_anode = np.atleast_1d(c.V_anode).astype(float)
    V_cathode = np.atleast_1d(c.V_cathode).astype(float)
    
    # Handle MATLAB column-major transposition for 2D arrays (nodes vs bias points)
    n_all = c.n if c.n.ndim > 1 else c.n.reshape(-1, 1)
    if n_all.shape[0] != num_nodes and n_all.shape[1] == num_nodes:
        n_all = n_all.T

    p_all = c.p if c.p.ndim > 1 else c.p.reshape(-1, 1)
    if p_all.shape[0] != num_nodes and p_all.shape[1] == num_nodes:
        p_all = p_all.T

    V_bias = V_anode - V_cathode if len(V_anode) == n_all.shape[1] else V_anode

    widths = [width_from_mesh(x_um, y_um, elements, n_all[:, i], p_all[:, i])
              for i in range(n_all.shape[1])]
    return np.array(V_bias), np.array(widths)


def process_folder(pattern):
    """One bias point per file (manual-loop .lsf output)."""
    V_bias, widths = [], []
    for f in sorted(glob.glob(pattern)):
        V, w = process_file(f)   # each file has exactly one bias point
        V_bias.append(V[0]); widths.append(w[0])
    order = np.argsort(V_bias)
    return np.array(V_bias)[order], np.array(widths)[order]

In [4]:
# --- pick ONE depending on which .lsf you ran ---
V_bias, widths = process_file('charge_sweep.mat')

for V, w in zip(V_bias, widths):
    print(f'V = {V:+.3f} V    ->    depletion width = {w:.1f} nm')

plt.figure(figsize=(5, 4))
plt.plot(V_bias, widths, 'o-')
plt.xlabel('Bias V_anode - V_cathode (V)')
plt.ylabel('Depletion width (nm)')
plt.title('Depletion width vs. bias')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('width_vs_bias.png', dpi=150)

V = -1.500 V    ->    depletion width = 284.6 nm
V = -1.425 V    ->    depletion width = 279.6 nm
V = -1.350 V    ->    depletion width = 275.6 nm
V = -1.275 V    ->    depletion width = 271.1 nm
V = -1.200 V    ->    depletion width = 265.6 nm
V = -1.125 V    ->    depletion width = 260.1 nm
V = -1.050 V    ->    depletion width = 253.6 nm
V = -0.975 V    ->    depletion width = 247.1 nm
V = -0.900 V    ->    depletion width = 239.6 nm
V = -0.825 V    ->    depletion width = 232.1 nm
V = -0.750 V    ->    depletion width = 226.1 nm
V = -0.675 V    ->    depletion width = 221.1 nm
V = -0.600 V    ->    depletion width = 215.1 nm
V = -0.525 V    ->    depletion width = 208.6 nm
V = -0.450 V    ->    depletion width = 202.1 nm
V = -0.375 V    ->    depletion width = 193.6 nm
V = -0.300 V    ->    depletion width = 185.1 nm
V = -0.225 V    ->    depletion width = 175.6 nm
V = -0.150 V    ->    depletion width = 167.6 nm
V = -0.075 V    ->    depletion width = 159.6 nm
V = +0.000 V    ->  